In [0]:
from pyspark.sql.functions import coalesce, try_to_date, regexp_replace, col, when

def process_orders(spark):
    # Read Bronze orders table
    orders_df = spark.read.parquet("/Volumes/workspace/default/medallion_data/output/bronze/orders/")
    
    # Clean Dates
    orders_df = orders_df.withColumn(
        "order_date",
        coalesce(
            try_to_date(col("order_date"), "yyyy-MM-dd"),
            try_to_date(col("order_date"), "MM/dd/yyyy"),
            try_to_date(col("order_date"), "dd-MM-yyyy")
        )
    )
    
    # Clean Amounts
    orders_df = orders_df.withColumn(
        "amount",
        when(regexp_replace(col("amount"), "[^0-9.]", "") == "", None)
        .otherwise(regexp_replace(col("amount"), "[^0-9.]", "").cast("double"))
    )
    
    # Join Dimensions
    products_df = spark.read.parquet("/Volumes/workspace/default/medallion_data/output/silver/products")
    customers_df = spark.read.parquet("/Volumes/workspace/default/medallion_data/output/silver/customers")
    
    joined_df = (
        orders_df
        .join(products_df, orders_df.product_id == products_df.product_id, "left")
        .join(customers_df, orders_df.customer_id == customers_df.customer_id, "left")
        .select(
            col("order_id"),
            col("order_date"),
            col("amount"),
            orders_df.product_id,
            orders_df.customer_id
        )
    )
    
    # Write to Silver orders table
    (
        joined_df
        .write
        .mode("append")
        .format("parquet")
        .save("/Volumes/workspace/default/medallion_data/output/silver/orders")
    )

In [0]:
process_orders(spark)